In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Load Dataset
df_phish = pd.read_csv('PhiUSIIL_Phishing_URL_Dataset.csv')

# 2. Data Cleaning and Feature Dropping
# Dropping useless identifier columns that offer zero predictive value. 
columns_to_drop = ['FILENAME'] 
df_phish.drop(columns=[col for col in columns_to_drop if col in df_phish.columns], inplace=True)

# Checking and removing duplicate rows
initial_rows = len(df_phish)
df_phish.drop_duplicates(inplace=True)
print(f"Dropped {initial_rows - len(df_phish)} duplicate rows.")

# 3. Missing Value Imputation
# Properly imputing missing values. Numerical features use median; categorical use mode.
num_cols = df_phish.select_dtypes(include=['int64', 'float64']).columns.drop('label', errors='ignore')
cat_cols = df_phish.select_dtypes(include=['object']).columns

imputer_num = SimpleImputer(strategy='median')
df_phish[num_cols] = imputer_num.fit_transform(df_phish[num_cols])

# 4. EDA and Skewness Profiling
# Applying log transformations to heavily skewed features (skew > 2) to normalize statistical distributions.
skewed_phish = df_phish[num_cols].apply(lambda x: x.skew()).sort_values(ascending=False)
high_skew_phish = skewed_phish[skewed_phish > 2].index
print(f"Applying Log-Transform to Highly Skewed Features: {list(high_skew_phish)}")

for feature in high_skew_phish:
    df_phish[feature] = np.log1p(df_phish[feature])

# 5. Encoding Categorical Predictors
# Prior to VIF calculation, categorical strings must be one-hot encoded.
df_phish = pd.get_dummies(df_phish, columns=cat_cols, drop_first=True)

# Define X and y (label: 1 = Legitimate, 0 = Phishing)
X_phish = df_phish.drop(columns=['label'])
y_phish = df_phish['label']

# 6. Multicollinearity Handling (VIF Calculation)
# Calculating the Variance Inflation Factor and dropping highly collinear predictors (threshold > 10).
def calculate_vif(X_df, threshold=10.0):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_df.columns
    # Handling potential division by zero and matrix instability
    vif_data["VIF"] = [variance_inflation_factor(X_df.values, i) for i in range(len(X_df.columns))]
    
    collinear_features = vif_data[vif_data["VIF"] > threshold]["feature"].tolist()
    return collinear_features

print("Calculating VIF...")
features_to_drop = calculate_vif(X_phish, threshold=10.0)
print(f"Dropping highly collinear features: {features_to_drop}")

X_phish_clean = X_phish.drop(columns=features_to_drop)

# 7. Train-Test Split & Feature Scaling
# Continuous data must be appropriately scaled to ensure proper regression convergence.
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_phish_clean, y_phish, test_size=0.20, random_state=42)

scaler_p = StandardScaler()
X_train_p_scaled = scaler_p.fit_transform(X_train_p)
X_test_p_scaled = scaler_p.transform(X_test_p)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, f1_score, classification_report

# 1. Scikit-Learn Logistic Regression Implementation
# Implementing Logistic Regression via Scikit-Learn.
# Class weights are balanced to stabilize classification between differing node occurrences.
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_p_scaled, y_train_p)

# Extract precise prediction probabilities to feed into the custom threshold evaluation.
y_probs = log_reg.predict_proba(X_test_p_scaled)[:, 1]

# 2. Custom Evaluation Script to Tune Decision Threshold
# This script dynamically iterates over decision thresholds to optimize performance based on the F1-Score.
def tune_decision_threshold(y_true, y_probabilities):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probabilities)
    
    # Calculate F1 scores across the entire continuum of thresholds
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10) # 1e-10 avoids division by zero errors
    
    # Locate the optimal index that yields the maximum possible F1-Score
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    optimal_f1 = f1_scores[optimal_idx]
    
    print("--- Custom Decision Threshold Tuning ---")
    print(f"Optimal Decision Threshold (τ): {optimal_threshold:.4f}")
    print(f"Maximum F1-Score Achieved:      {optimal_f1:.4f} (Target: > 0.98)")
    
    # Plot Precision-Recall Trade-off against custom decision boundaries
    plt.figure(figsize=(8, 5))
    plt.plot(thresholds, precisions[:-1], 'b--', label='Precision')
    plt.plot(thresholds, recalls[:-1], 'g-', label='Recall')
    plt.plot(thresholds, f1_scores[:-1], 'r-', label='F1 Score')
    plt.axvline(x=optimal_threshold, color='k', linestyle=':', label=f'Optimal τ = {optimal_threshold:.2f}')
    plt.xlabel('Decision Threshold (τ)')
    plt.ylabel('Performance Metric Score')
    plt.title('Precision-Recall Trade-off Tuning Profile')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return optimal_threshold, optimal_f1

optimal_tau, final_f1 = tune_decision_threshold(y_test_p, y_probs)

# Apply custom decision threshold for the definitive model predictions
custom_predictions = (y_probs >= optimal_tau).astype(int)

print("\nFinal Classification Report (Using Custom Tuned Threshold):")
print(classification_report(y_test_p, custom_predictions))